# Evaluating Generated Text, and Its Circularity

**Computational Text Analysis Pack · Module 27 — Generative Models in DH Research**

*Technique:* BLEU/ROUGE and their limits; LLM-as-judge and why it is circular

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/14-generative-llms/27c_evaluating_generated_text.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

If a model generates a summary, how do you know it is any good? The field's
answers are weaker than their ubiquity suggests, and knowing exactly how they
are weak is what stops you reporting a meaningless number.

## The tooling

**HuggingFace `transformers`** runs open models locally, including the decoding
strategies below. **vLLM** and **llama.cpp** are the usual serving layers when
you need throughput. Hosted APIs — OpenAI, Anthropic, Cohere — are what most
researchers actually reach for.

In CourseHub these cells go through a **gateway** that holds the credential
server-side and routes to open models. In Colab you supply your own key, or the
notebook falls back to recorded output. No API key appears in any cell of this
course, because a notebook cell is readable by whoever runs it.

**The cost question is a methods question.** Generating over a 5,000-document
corpus is not free and not fast, and unlike a frequency count it is not
deterministic — two runs give two answers. Budget and reproducibility both bite.

In [ ]:
import sys, os, json, textwrap

IN_COLAB = "google.colab" in sys.modules
LLM_BACKEND = "recorded"
_chat = None

if not IN_COLAB:
    try:
        sys.path.insert(0, "/app")
        from utils.llm_gateway import gateway_chat as _gw
        def _chat(messages, model="gpt-oss-20b", max_tokens=300):
            # allow_fallback=False keeps us on the local provider; the fallback
            # is billed per token and a classroom can run a lot of tokens.
            txt, meta = _gw(messages, model=model, max_tokens=max_tokens,
                            allow_fallback=False)
            return txt, meta.get("served_model", model)
        _chat([{"role": "user", "content": "ok"}], max_tokens=5)
        LLM_BACKEND = "coursehub-gateway"
    except Exception as exc:
        _why = str(exc)[:80]
else:
    key = os.environ.get("OPENAI_API_KEY", "")
    if key:
        def _chat(messages, model="gpt-4o-mini", max_tokens=300):
            import requests
            r = requests.post("https://api.openai.com/v1/chat/completions",
                              headers={"Authorization": f"Bearer {key}"},
                              json={"model": model, "messages": messages,
                                    "max_tokens": max_tokens}, timeout=60)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"], model
        LLM_BACKEND = "colab-openai"
    else:
        _why = "no OPENAI_API_KEY set in this Colab session"

def ask(prompt, model=None, max_tokens=300, system=None):
    """One prompt in, one string out — or a recorded answer when offline."""
    if _chat is None:
        return None, "recorded"
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    kw = {"max_tokens": max_tokens}
    if model:
        kw["model"] = model
    return _chat(msgs, **kw)

print(f"LLM backend: {LLM_BACKEND}")
if LLM_BACKEND == "recorded":
    print(f"   ({_why})")
    print("   Cells below print outputs recorded when this notebook was written,")
    print("   clearly marked, so the lesson still works without a model.")
elif LLM_BACKEND == "colab-openai":
    print("   using your own key from this Colab session")
else:
    print("   using the CourseHub gateway; no key needed and none is exposed here")

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

## BLEU and ROUGE: overlap, not quality

Both compare generated text against a human reference by counting shared
n-grams. BLEU leans on precision, ROUGE on recall. Both are blind to meaning —
which we can demonstrate rather than assert.

In [ ]:
def ngrams(toks, n):
    return Counter(tuple(toks[i:i+n]) for i in range(len(toks) - n + 1))

def bleu_ish(cand, ref, max_n=4):
    c, r = tokenise(cand), tokenise(ref)
    if not c:
        return 0.0
    precs = []
    for n in range(1, max_n + 1):
        cn, rn = ngrams(c, n), ngrams(r, n)
        overlap = sum(min(v, rn[k]) for k, v in cn.items())
        precs.append(overlap / max(sum(cn.values()), 1))
    if min(precs) == 0:
        return 0.0
    bp = min(1.0, np.exp(1 - len(r) / max(len(c), 1)))
    return bp * float(np.exp(np.mean(np.log(precs))))

def rouge_1(cand, ref):
    c, r = Counter(tokenise(cand)), Counter(tokenise(ref))
    overlap = sum(min(v, r[k]) for k, v in c.items())
    return overlap / max(sum(r.values()), 1)

REF = "The government announced new funding for rural health clinics this week."
CANDIDATES = [
    ("near-copy",        "The government announced new funding for rural health clinics this week."),
    ("correct paraphrase","Rural health centres will receive fresh state money, officials said."),
    ("fluent but WRONG",  "The government announced new funding for rural football clubs this week."),
    ("word salad",        "clinics week rural the funding announced government health new for this."),
]
print(f"{'candidate':<22}{'BLEU':>8}{'ROUGE-1':>10}")
print("-" * 42)
for name, cand in CANDIDATES:
    print(f"{name:<22}{bleu_ish(cand, REF):>8.3f}{rouge_1(cand, REF):>10.3f}")

In [ ]:
print("Read those four rows against each other:\n")
print("  * The CORRECT PARAPHRASE scores near zero. It says the right thing in")
print("    different words, and n-gram overlap cannot see that.")
print("  * 'football clubs' scores almost as high as the near-copy. One word")
print("    changed, the meaning is now false, the metric barely moved.")
print("  * WORD SALAD scores respectably on ROUGE-1, which ignores order.")
print()
print("A metric that rewards a false sentence and punishes a true one is not")
print("measuring quality. Report BLEU/ROUGE only where they are conventional")
print("(machine translation), and never as evidence a summary is accurate.")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
names = [n for n, _ in CANDIDATES]
ax.bar(np.arange(len(names)) - 0.2, [bleu_ish(c, REF) for _, c in CANDIDATES],
       width=.4, label="BLEU")
ax.bar(np.arange(len(names)) + 0.2, [rouge_1(c, REF) for _, c in CANDIDATES],
       width=.4, label="ROUGE-1")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20, ha="right", fontsize=8)
ax.legend(); ax.set_ylabel("score")
ax.set_title("The metrics disagree with meaning")
plt.tight_layout()

## LLM-as-judge, and the circle it closes

The current fashion is to have a model score the output. It correlates better
with human judgement than BLEU does — and it has a problem no amount of
correlation fixes.

**The judge shares the generator's blind spots.** A model that thinks *The
Palm-Wine Drinkard* is by Achebe will happily approve a summary that says so.
On exactly the material where you most need an independent check — minor
literatures, African languages, anything thin in pretraining — the judge is
least independent.

In [ ]:
JUDGE_SYSTEM = ("You are grading a factual answer. Reply with a single integer "
                "1-5 for accuracy, then one short sentence of justification.")
CLAIM = ("The Palm-Wine Drinkard is a 1953 novel by Chinua Achebe about a man "
         "who journeys to find his dead palm-wine tapster.")

if LLM_BACKEND != "recorded":
    out, served = ask(f"Statement: {CLAIM}\nGrade it.", system=JUDGE_SYSTEM,
                      max_tokens=120)
    print(f"judge ({served}):", " ".join(out.split())[:260])
else:
    print("RECORDED: the judge graded this statement highly, repeating the same")
    print("misattribution the generator made in notebook 27a.")
print()
print("Ground truth: Amos Tutuola, 1952. The plot detail is right; the author")
print("and the year are wrong. A judge sharing the generator's training data")
print("is not an independent check — it is the same error, counted twice.")

## What to actually do

- **Human evaluation on a sample.** Slow, and still the only defensible option
  for a claim about accuracy. Use module 17's agreement statistics on it.
- **Task-specific checks.** If the model extracts dates, check the dates against
  the text. Verifiable sub-claims beat holistic scores.
- **Report the protocol, not just the number.** Which model judged, which
  version, which prompt, how many items, and who checked the sample.
- **Prefer grounded architectures.** Notebook 27b's citations can be opened.
  That is worth more than any score.

### 🔧 Your turn 1

Take five generated summaries of texts you know well, grade them yourself, then have a model grade them. Where do you disagree?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Disagreements cluster on exactly the items where the model lacks knowledge, and
they run in one direction: the judge is too generous about fluent text
containing a factual error, because fluency is what it can assess and fact is
what it cannot. Reporting your own agreement with the judge — a kappa from
module 17 — is a far stronger methods statement than reporting the judge's mean
score, and it costs an afternoon.

</details>